# Development of an AI Product Managemer

To develop an AI-powered system that functions as a Product Manager by analyzing digital
products, providing critical evaluations, suggesting improvements, and generating prioritized
user stories for implementation.


In this project, we are building a multi-agent system using the crewai library to analyze an image of a B2B digital menu startup. The system consists of four agents:

* Image Description Agent: Provides a detailed description of the image.
* UX Critique Agent: Critically analyzes the image based on the description.
* UX Suggestion Agent: Offers actionable suggestions to improve the image design.
* AI Product Manager: Writes user stories and prioritizes suggestions based on customer feedback.

# Setup

We start by setting up the working directory and installing the necessary libraries.

In [1]:
# Install the 'crewai' library and its tools, along with 'openai'
!pip install crewai
!pip install openai
!pip install 'crewai[tools]'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.8/131.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 628.3/628.3 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.0/468.0 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.1/70.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# api key
genai_apikey = 'your open ai api key'
api_key = genai_apikey

In [2]:
# Import essential libraries for image processing and agent creation
import os
from PIL import Image  # For image manipulation
from crewai_tools import VisionTool  # Specialized tool for image analysis
from crewai import Agent, Task, Crew, Process  # Core components of the crewai library
from langchain_openai import ChatOpenAI  # Interface to interact with OpenAI's language models
from IPython.display import display, Markdown  # For displaying outputs in Jupyter notebooks

We import PIL for image handling, crewai components for agent management, and langchain_openai to interface with OpenAI's GPT-4 model. The VisionTool is a custom tool for image analysis.

In [3]:
# Set the OpenAI API key as an environment variable
os.environ['OPENAI_API_KEY'] = api_key

In [4]:
# Initialize the vision tool for image analysis
vision_tool = VisionTool()

By setting the API key as an environment variable, we ensure it's accessible throughout the program. The VisionTool will allow our agents to process and analyze images.

In [5]:
# Load the Image using the Pillow library
image_path = "translation.png"
image = Image.open(image_path)

The image translation.png is opened and stored in the image variable for later use.

# AGENT 1 - Image Description Agent

We create the first agent responsible for describing the image.

In [6]:
# Build the description agent with its role, goal, and backstory
description_agent = Agent(
    role="Image Description Agent",
    goal=f"Fully describe the digital image ({image_path}), of a B2B Digital Menu startup, including its visible elements, design, and intended purpose.",
    backstory="You are responsible for analyzing images and describing their purpose in detail.",
    verbose=True,
    tools=[vision_tool],
    llm=ChatOpenAI(model_name="gpt-4o", temperature=0.8)
)


In [7]:
# Create the description task assigned to the description agent
description_task = Task(
    description="Identify and fully describe the digital image and explain its purpose.",
    expected_output="A complete description of the image and its purpose.",
    agent=description_agent
)

The Image Description Agent uses the VisionTool to analyze the image and provides a detailed description. We set the language model to GPT-4 with a temperature of 0.8 to allow for creative outputs.

# AGENT 2 - Critique Agent

We create the second agent to critique the image based on the description.

In [8]:
# Build the critique agent with its role, goal, and backstory
critique_agent = Agent(
    role="UX Critique Agent",
    goal=f"Critique the image {image_path} based on its description and intended purpose provided by the Image Description Agent.",
    backstory="You critically evaluate images, specifically UX designs, and point out flaws, weaknesses, and areas of improvement.",
    verbose=True,
    tools=[vision_tool],
    llm=ChatOpenAI(model_name="gpt-4o", temperature=0.8)
)

In [9]:
# Create the critique task assigned to the critique agent, using the description task as context
critique_task = Task(
    description="Critically analyze the image based on its description and intended purpose.",
    expected_output="A complete critique of the image, highlighting design flaws and areas of improvement.",
    agent=critique_agent,
    context=[description_task]
)

The UX Critique Agent evaluates the image using the description from the first agent. The context parameter allows the agent to access the output of the description_task.

# AGENT 3 - UX Suggestion

We create the third agent to provide suggestions for improving the image.

In [10]:
# Create the UX suggestion agent with its role, goal, and backstory
ux_agent = Agent(
    role="UX Suggestion Agent",
    goal=f"Provide design and layout suggestions for the image {image_path} based on the context from the Image Description Agent and UX Critique Agent.",
    backstory="You specialize in providing actionable suggestions to improve the design of website images.",
    verbose=True,
    tools=[vision_tool],
    llm=ChatOpenAI(model_name="gpt-4o", temperature=0.8)
)

In [11]:
# Create the UX suggestion task assigned to the UX suggestion agent, using previous tasks as context
ux_task = Task(
    description="Provide suggestions for improving the image design and layout, based on the context from the description and critique agents.",
    expected_output="A list of actionable suggestions for improving the image design and layout based on the image's purpose and critiques.",
    agent=ux_agent,
    context=[description_task, critique_task]
)

The UX Suggestion Agent leverages the outputs from both the description and critique agents to generate improvement suggestions.

# AGENT 4 - AI Product Manager

We create the fourth agent to write user stories and prioritize suggestions.

In [12]:
# Define the AI Product Manager Agent with its role, goal, and backstory
pm_agent = Agent(
    role="AI Product Manager",
    goal=f"Write user stories based on the suggestions from the UX agent for the image {image_path} and prioritize the suggestions based on probable customer feedback.",
    backstory="You act as a product manager for a digital company, prioritizing suggestions and creating user stories to guide improvements.",
    verbose=True,
    tools=[vision_tool],
    llm=ChatOpenAI(model_name="gpt-4o", temperature=0.8)
)

In [13]:
# Create a task for the AI Product Manager, using all previous tasks as context
pm_task = Task(
    description="Write user stories based on the suggestions from the UX agent for the image and prioritize the suggestions based on probable customer feedback.",
    expected_output="A list of prioritized improvements based on expected impact on the customer, and the user stories.",
    agent=pm_agent,
    context=[description_task, critique_task, ux_task]
)

The AI Product Manager consolidates the suggestions and critiques to create actionable user stories, prioritizing them based on customer impact.

# Run the AI Product Manager

We now define the crew and run the tasks sequentially.

In [14]:
# Define the crew with all agents and tasks, set to run sequentially
crew = Crew(
    agents=[description_agent, critique_agent, ux_agent, pm_agent],
    tasks=[description_task, critique_task, ux_task, pm_task],
    verbose=True,
    process=Process.sequential
)

# Kick off the crew to start processing the tasks
result = crew.kickoff()

# Agent: Image Description Agent
## Task: Identify and fully describe the digital image and explain its purpose.


# Agent: Image Description Agent
## Using tool: Vision Tool
## Tool Input: 
"{\"image_path_url\": \"translation.png\"}"
## Tool Output: 
The image shows a digital interface for a menu translation tool, likely for a restaurant management system. 

Key elements include:

- A sidebar with options such as "Restaurants," "Edit menu," "Print menu," "Categories," and "Translation Center."
- A main section titled "Translate - Menu," indicating the sections for food ("Comida") and specific categories like "Entradas" (appetizers) and "Pizzas."
- Listed food items under "Entradas" (e.g., "Bruschetta al Pomodoro," "Sopa de Mexilhão") and "Pizzas" (e.g., "Margherita," "Diavola").
- A section for language selection, allowing users to choose between Portuguese and English.
- Input fields for translating menu items with a "Save" button next to each language.

Overall, it's a tool designed

The Crew object manages all agents and tasks. By setting the process to Process.sequential, we ensure that each task is completed before the next one starts, maintaining the context flow.

Finally, we extract and display the outputs from each agent. We loop through each task's output and display it using Markdown formatting for better readability in the Jupyter notebook.

In [15]:
# extract and display the output of each agent
for idx, task_output in enumerate(result.tasks_output):
  display(Markdown(f"### Agent {idx+1}: {task_output.agent}\n{task_output.raw}"))


### Agent 1: Image Description Agent
The image, "translation.png," depicts a digital interface for a menu translation tool, which appears to be part of a B2B digital menu startup. The purpose of this tool is to facilitate the translation of restaurant menus for businesses that cater to a multilingual clientele. 

The interface prominently displays several key elements:

1. **Sidebar Menu**: On the left side, there is a vertical sidebar with various navigation options such as "Restaurants," "Edit menu," "Print menu," "Categories," and "Translation Center." These options suggest that the tool offers comprehensive menu management services beyond just translation.

2. **Main Translation Section**: The central part of the interface is titled "Translate - Menu," indicating the area where the translation process occurs. This section is divided into different food categories, such as "Comida" (Food), with subcategories like "Entradas" (Appetizers) and "Pizzas."

3. **Food Items**: Examples of food items listed under these categories include "Bruschetta al Pomodoro" and "Sopa de Mexilhão" under "Entradas," and "Margherita" and "Diavola" under "Pizzas." These items are presented alongside translation fields.

4. **Language Selection**: There is a section dedicated to language selection, where users can translate menu items from Portuguese to English. This includes input fields for entering translations and a "Save" button for each language option, suggesting that changes can be saved individually.

5. **User Interaction**: The tool is interactive, allowing users to input translations directly and presumably update or modify them as needed. The presence of a "Save" button indicates that users can save their translations, ensuring that menu updates are retained.

Overall, the digital interface is designed to help restaurant businesses efficiently manage and translate their menus, enhancing their ability to serve a diverse customer base by offering menu options in multiple languages. This tool is likely intended for use by restaurant managers or staff responsible for menu curation and updates.

### Agent 2: UX Critique Agent
The "translation.png" interface design for a B2B digital menu translation tool possesses several strengths and areas that need improvement to enhance its usability and efficiency. Here is a detailed critique based on the image's description and intended purpose:

1. **Strengths:**
   - **Comprehensive Navigation:** The sidebar menu provides a wide range of options such as "Restaurants," "Edit menu," "Print menu," "Categories," and "Translation Center," indicating a robust set of features beyond just translation. This caters well to restaurant managers who require more than just language translation capabilities.
   - **Language Flexibility:** The language selection feature that allows translation from Portuguese to English, with separate "Save" buttons for each language, suggests an understanding of the need for flexibility and precision in multilingual menu management.
   - **Category Organization:** The division of the main translation section into clear food categories (e.g., "Comida," "Entradas," "Pizzas") helps users quickly locate and manage specific menu items, which is efficient for large menus.

2. **Design Flaws and Areas for Improvement:**
   - **Visual Hierarchy and Clarity:** While the interface appears to have a structured layout, there may be a need to enhance the visual hierarchy to guide users more intuitively through the translation process. For example, highlighting active sections or using contrasting colors for essential buttons could improve user focus and interaction.
   - **Feedback Mechanisms:** The interface lacks visible feedback mechanisms that inform users when their translations are successfully saved or if an error occurs. Implementing confirmation messages or status indicators would enhance user confidence and usability.
   - **Scalability of Input Fields:** The input fields for translation should be scalable or expandable to accommodate longer text entries, ensuring that users can view their entire translation without needing to scroll horizontally.
   - **User Guidance and Help:** The interface would benefit from integrated help features or tooltips, especially for users who may be unfamiliar with the translation tool. This could include guidance on how to best utilize the translation fields and save functions.
   - **Consistency in Iconography and Labels:** Ensuring consistency in icons and labels across the interface can reduce cognitive load and improve the ease of navigation, especially if the tool is part of a larger suite of restaurant management applications.

3. **Potential Enhancements:**
   - **Batch Processing Options:** For efficiency, especially in large menus, consider adding options for batch processing of translations, where multiple items can be translated and saved with fewer steps.
   - **Mobile Responsiveness:** If not already designed for, ensuring that the interface is mobile-responsive can significantly enhance the tool's utility for restaurant staff who may need to make updates on-the-go.
   - **Personalization Features:** Allowing users to customize the interface, such as setting default languages or frequently accessed categories, could improve user experience and efficiency.

By addressing these areas, the "translation.png" interface could offer an even more powerful and user-friendly solution for restaurant businesses managing multilingual menus.

### Agent 3: UX Suggestion Agent
Based on the description and critique of the "translation.png" image, here are actionable suggestions for improving the image design and layout:

1. **Enhance Visual Hierarchy:**
   - Use contrasting colors or bold fonts for essential buttons like "Save" to draw user attention.
   - Highlight active sections of the sidebar or main translation area to guide user navigation intuitively.

2. **Implement Feedback Mechanisms:**
   - Introduce confirmation messages or status indicators when translations are successfully saved or if an error occurs. This will provide users with immediate feedback on their actions.

3. **Improve Input Field Scalability:**
   - Ensure that input fields for translations are adjustable or expandable, allowing users to view and edit longer text entries without horizontal scrolling.

4. **Add User Guidance and Help:**
   - Integrate tooltips or help icons that offer quick guidance on using the translation fields and save functions, particularly for new users who may find the interface unfamiliar.

5. **Ensure Iconography and Label Consistency:**
   - Maintain consistent icon styles and labels throughout the interface to reduce cognitive load and improve navigation.

6. **Introduce Batch Processing Options:**
   - Provide options for batch processing to enable users to translate and save multiple items simultaneously, enhancing efficiency for large menus.

7. **Ensure Mobile Responsiveness:**
   - Design the interface to be mobile-responsive, allowing restaurant staff to make updates on-the-go, thus increasing the tool's utility and flexibility.

8. **Personalization Features:**
   - Allow users to customize their interface by setting default languages or frequently accessed categories, which could streamline the user experience.

By implementing these improvements, the "translation.png" interface can become a more effective and user-friendly tool for restaurant businesses managing multilingual menus.

### Agent 4: AI Product Manager
**Prioritized Improvements and User Stories:**

1. **Implement Feedback Mechanisms**
   - *Priority*: High
   - *User Story*: As a restaurant manager, I want to receive confirmation messages or status indicators when translations are successfully saved or when errors occur, so that I can feel confident that my updates have been processed correctly.

2. **Enhance Visual Hierarchy**
   - *Priority*: High
   - *User Story*: As a user, I want essential buttons like "Save" to be highlighted with contrasting colors or bold fonts, so that I can quickly and easily locate and use them, improving my overall navigation experience.

3. **Ensure Mobile Responsiveness**
   - *Priority*: High
   - *User Story*: As a restaurant staff member, I need the interface to be mobile-responsive, so I can make menu updates on-the-go, thus increasing the flexibility and utility of the tool.

4. **Improve Input Field Scalability**
   - *Priority*: Medium
   - *User Story*: As a translator, I want the input fields to be expandable to accommodate longer text entries, so I can easily view and edit translations without horizontal scrolling.

5. **Add User Guidance and Help**
   - *Priority*: Medium
   - *User Story*: As a new user, I want integrated tooltips or help icons that offer quick guidance on using the translation fields and save functions, so I can efficiently learn how to use the tool without external help.

6. **Introduce Batch Processing Options**
   - *Priority*: Medium
   - *User Story*: As a restaurant manager with a large menu, I want to process multiple translations at once, so I can save time and increase efficiency.

7. **Ensure Iconography and Label Consistency**
   - *Priority*: Low
   - *User Story*: As a user, I want consistent icon styles and labels throughout the interface, so I experience reduced cognitive load and can navigate more easily.

8. **Personalization Features**
   - *Priority*: Low
   - *User Story*: As a frequent user, I want to customize the interface by setting default languages or frequently accessed categories, so I can streamline my tasks and improve my workflow. 

These user stories address the main areas of improvement identified in the critique, with an emphasis on enhancing usability, efficiency, and user experience. The prioritization reflects the potential impact on customer satisfaction and the overall functionality of the tool.